In [3]:
%pwd

'd:\\Tipto\\Natural-Language-Processing'

In [2]:
import os 
os.chdir('../')

In [8]:
import pandas as pd 
data = pd.read_csv("Data/ner.csv")
data.head()

,Sentence #,Sentence,POS,Tag
0,Sentence: 1,Thousands of demonstrators have marched throug...,"['NNS', 'IN', 'NNS', 'VBP', 'VBN', 'IN', 'NNP'...","['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', '..."
1,Sentence: 2,Families of soldiers killed in the conflict jo...,"['NNS', 'IN', 'NNS', 'VBN', 'IN', 'DT', 'NN', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
2,Sentence: 3,They marched from the Houses of Parliament to ...,"['PRP', 'VBD', 'IN', 'DT', 'NNS', 'IN', 'NN', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
3,Sentence: 4,"Police put the number of marchers at 10,000 wh...","['NNS', 'VBD', 'DT', 'NN', 'IN', 'NNS', 'IN', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
4,Sentence: 5,The protest comes on the eve of the annual con...,"['DT', 'NN', 'VBZ', 'IN', 'DT', 'NN', 'IN', 'D...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."


In [5]:
data.columns

Index(['Sentence #', 'Sentence', 'POS', 'Tag'], dtype='object')

In [9]:
# drop all columns except Sentence and Tag, cause we are only concerned about NER tagging.
data.drop(columns = ['Sentence #', 'POS'] , axis = 1, inplace = True)

In [10]:
data.head()

,Sentence,Tag
0,Thousands of demonstrators have marched throug...,"['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', '..."
1,Families of soldiers killed in the conflict jo...,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
2,They marched from the Houses of Parliament to ...,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
3,"Police put the number of marchers at 10,000 wh...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
4,The protest comes on the eve of the annual con...,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."


In [11]:
data.iloc[0]

Sentence    Thousands of demonstrators have marched throug...
Tag         ['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', '...
Name: 0, dtype: object

In [21]:
data['Tag'][0]

"['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-gpe', 'O', 'O', 'O', 'O', 'O']"

In [22]:
import ast
tags_list = ast.literal_eval(data['Tag'][0])
print(tags_list)
print(type(tags_list))

['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-gpe', 'O', 'O', 'O', 'O', 'O']
<class 'list'>


In [24]:
data['Tag'] = data['Tag'].apply(ast.literal_eval)

In [25]:
labels = set()
i = 0
for tags in data['Tag']:
    for tag in tags:
        labels.add(tag)

labels

{'B-art',
 'B-eve',
 'B-geo',
 'B-gpe',
 'B-nat',
 'B-org',
 'B-per',
 'B-tim',
 'I-art',
 'I-eve',
 'I-geo',
 'I-gpe',
 'I-nat',
 'I-org',
 'I-per',
 'I-tim',
 'O'}

In [28]:
# map all labels with a unique id and vice versa
id_to_labels = {key : val for key , val in enumerate(sorted(labels))}
labels_to_id= {val : key for key , val in id_to_labels.items()}
labels_to_id

{'B-art': 0,
 'B-eve': 1,
 'B-geo': 2,
 'B-gpe': 3,
 'B-nat': 4,
 'B-org': 5,
 'B-per': 6,
 'B-tim': 7,
 'I-art': 8,
 'I-eve': 9,
 'I-geo': 10,
 'I-gpe': 11,
 'I-nat': 12,
 'I-org': 13,
 'I-per': 14,
 'I-tim': 15,
 'O': 16}

In [27]:
labels_to_id[0]

'B-art'

### Import BERT tokenizer for the tokenization.
  - Here we are using `bert-base-cased` tokenizer.

In [29]:
from transformers import BertTokenizerFast
model_checkpoint = "bert-base-cased"
tokenizer = BertTokenizerFast.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [30]:
sample_text = data['Sentence'][0]
sample_text

'Thousands of demonstrators have marched through London to protest the war in Iraq and demand the withdrawal of British troops from that country .'

In [44]:
sample_tokenized = tokenizer(
    sample_text,
    padding = 'max_length',
    truncation = True,
    max_length = 512,
    return_tensors = 'pt'
)
print(sample_tokenized)

{'input_ids': tensor([[  101, 26159,  1104,  8568,  4487,  5067,  1138,  9639,  1194,  1498,
          1106,  5641,  1103,  1594,  1107,  5008,  1105,  4555,  1103, 10602,
          1104,  1418,  2830,  1121,  1115,  1583,   119,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,  

In [37]:
print(tokenizer.decode(sample_tokenized.input_ids[0]))

[CLS] Thousands of demonstrators have marched through London to protest the war in Iraq and demand the withdrawal of British troops from that country. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PA

### Now we have to assign a label to every words.
---
  - But BERT tokenizer added new special tokens like [CLS],[SEP],[PAD],
  we have to handle this.
  - Also BERT uses WordPiece tokenization, so some words are splited but we have to assign the same label to all these splited part.
  - So for words which is splited, these all splited part should have the same label and the tokens which are not a part of main sentence words like the special token we have to give them None as label.
  - To all these stuffs we need a `word_ids`.

In [42]:
word_ids = sample_tokenized.word_ids()
print(tokenizer.convert_ids_to_tokens(sample_tokenized['input_ids'][0]))
print(word_ids)

['[CLS]', 'Thousands', 'of', 'demons', '##tra', '##tors', 'have', 'marched', 'through', 'London', 'to', 'protest', 'the', 'war', 'in', 'Iraq', 'and', 'demand', 'the', 'withdrawal', 'of', 'British', 'troops', 'from', 'that', 'country', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PA

### Define a tokenization + label alignmnent function.

In [51]:
data['Sentence'] = data['Sentence'].apply(lambda x: x.split())

In [62]:
data['Tag'] = data['Tag'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [63]:
tag2id = {tag: i for i, tag in enumerate(sorted(set(t for tags in data['Tag'] for t in tags)))}
data['Tag'] = data['Tag'].apply(lambda tags: [tag2id[t] for t in tags])

In [53]:
sentence = data['Sentence'][0]
tags = data['Tag'][0]
labels = [labels_to_id[t] for t in tags]
sentence

['Thousands',
 'of',
 'demonstrators',
 'have',
 'marched',
 'through',
 'London',
 'to',
 'protest',
 'the',
 'war',
 'in',
 'Iraq',
 'and',
 'demand',
 'the',
 'withdrawal',
 'of',
 'British',
 'troops',
 'from',
 'that',
 'country',
 '.']

In [71]:
def tokenize_and_align_labels(sentence, labels, tokenizer, max_length = 512):
    # tokenize the sentence
    tokenized_inputs = tokenizer(
        sentence,
        truncation = True,
        padding = 'max_length',
        max_length = max_length,
        return_tensors = None,
        is_split_into_words = True
    )
    
    word_ids = tokenized_inputs.word_ids()
    # align labels with tokens
    aligned_labels = []
    
    for word_idx in word_ids:
        if word_idx is None: # special token like [CLS],[PAD],[SEP]
            aligned_labels.append(-100) # label id -100 will be ignored by loss function
        elif word_idx < len(labels): 
            aligned_labels.append(labels[word_idx])
        else:
            # for subword tokens inside the same word, add the same label_id as prev
            # prev word index and current word index is same, take anyone
            aligned_labels.append(-100)
        
    tokenized_inputs['labels'] = aligned_labels
    return tokenized_inputs    

In [72]:
tokenized_inputs = tokenizer(
    sentence,
    truncation = True,
    padding = 'max_length',
    max_length = 512,
    return_tensors = None,
    is_split_into_words = True
)
word_ids = tokenized_inputs.word_ids()
print(sentence)
print(tokenized_inputs)
print(word_ids)

['Thousands', 'of', 'demonstrators', 'have', 'marched', 'through', 'London', 'to', 'protest', 'the', 'war', 'in', 'Iraq', 'and', 'demand', 'the', 'withdrawal', 'of', 'British', 'troops', 'from', 'that', 'country', '.']
{'input_ids': [101, 26159, 1104, 8568, 4487, 5067, 1138, 9639, 1194, 1498, 1106, 5641, 1103, 1594, 1107, 5008, 1105, 4555, 1103, 10602, 1104, 1418, 2830, 1121, 1115, 1583, 119, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [73]:
tokenized_data = data.apply(
    lambda row: tokenize_and_align_labels(
        sentence=row['Sentence'],
        labels=row['Tag'],
        tokenizer=tokenizer
    ),
    axis=1
)

In [74]:
tokenized_data

0        [input_ids, token_type_ids, attention_mask, la...
1        [input_ids, token_type_ids, attention_mask, la...
2        [input_ids, token_type_ids, attention_mask, la...
3        [input_ids, token_type_ids, attention_mask, la...
4        [input_ids, token_type_ids, attention_mask, la...
                               ...                        
47954    [input_ids, token_type_ids, attention_mask, la...
47955    [input_ids, token_type_ids, attention_mask, la...
47956    [input_ids, token_type_ids, attention_mask, la...
47957    [input_ids, token_type_ids, attention_mask, la...
47958    [input_ids, token_type_ids, attention_mask, la...
Length: 47959, dtype: object

### Reformat the Data
- BERT expects a dictionary of lists (or a Hugging Face Dataset object), not a Pandas Series of dictionaries.

In [75]:
from datasets import Dataset

In [76]:
# Convert the Series of dictionaries into a list of dictionaries, then to a Dataset
df_final = pd.DataFrame(tokenized_data.tolist())
dataset = Dataset.from_pandas(df_final)

In [77]:
# Split into train and test
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset['train']
test_dataset = dataset['test']

### Initialize the BERT Model for NER

In [78]:
from transformers import BertForTokenClassification, TrainingArguments,Trainer

In [79]:
model = BertForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels = len(id_to_labels),
    id2label = id_to_labels,
    label2id = labels_to_id
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Setup Data Collator and Metrices.

In [80]:
from transformers import DataCollatorForTokenClassification
import evaluate

In [81]:
data_collator = DataCollatorForTokenClassification(tokenizer = tokenizer)
metric = evaluate.load('seqeval')

In [82]:
import numpy as np 
def compute_metrics(p): 
    predictions , labels = p 
    predictions = np.argmax(predictions , axis = 2)
    
    # Remove ignored index (special tokens)
    true_predictions = [
        [id_to_labels[p] for (p , l) in zip(prediction , label) if l != -100]
        for prediction , label in zip(predictions , labels)
    ]
    true_labels = [
        [id_to_labels[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = metric.compute(
        predictions = true_predictions , references = true_labels
    )
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

### Define Training Arguments and Trainer.

In [90]:
import torch

In [91]:
training_args = TrainingArguments(
    output_dir = "./bert-ner-results",
    eval_strategy = "epoch",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 2,
    weight_decay = 0.01,
    report_to = 'none',
    fp16 = torch.cuda.is_available()
)

In [92]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    tokenizer = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
)

C:\Users\tipto\AppData\Local\Temp\ipykernel_31948\4036719813.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [93]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.140700,0.128762,0.816797,0.815913,0.816355,0.958869


KeyboardInterrupt: 